# 2장 1강: AARRR 프레임워크 개요와 단계별 분석 목적 — 실습문제

## 실습 목표

- Ravenstack의 사용자 여정을 AARRR 5단계로 구분할 수 있다.
- 각 단계의 분석 목적과 활용 가능한 데이터를 연결할 수 있다.
- 이벤트 로그의 사용자·행동·시점·속성 컬럼을 구분할 수 있다.
- 분석 질문에 맞게 퍼널·코호트·세그멘테이션 분석을 선택할 수 있다.
- 그룹별 Activation 지표를 비교하여 점검 대상을 찾을 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`
- `ravenstack_feature_usage.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

Ravenstack의 AARRR 단계는 다음과 같이 연결할 수 있습니다.

| 단계 | Ravenstack에서의 의미 | 활용 데이터 예시 |
|---|---|---|
| Acquisition | 고객사가 유입되어 가입함 | 가입일, 유입 경로 |
| Activation | 구독 후 제품 기능을 처음 사용함 | 기능 사용 기록 |
| Retention | 제품 사용과 구독을 계속 유지함 | 반복 사용 기록, 이탈 여부 |
| Referral | 기존 고객이 다른 고객을 추천함 | 추천 전송·추천 가입 이벤트 |
| Revenue | 유료 구독에서 반복 매출이 발생함 | MRR, 요금제 |

> 제공된 데이터에는 고객 추천 행동을 직접 기록한 이벤트가 없습니다. `referral_source`는 고객의 유입 경로이며, 기존 고객이 추천 행동을 했다는 사실을 직접 보여주는 Referral 이벤트와는 구분해야 합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 세 CSV 파일을 각각 `accounts`, `subscriptions`, `feature_usage`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`, `usage_date`를 날짜형으로 변환하세요.
6. `feature_usage`에서 이벤트 식별자인 `usage_id`의 중복 개수를 확인하세요.
7. 기능별 이벤트 기록 수와 `usage_count`의 기초 통계량을 확인하세요.

In [2]:
# 실습 준비 코드를 작성하세요.

import pandas as pd

# 1~2. 라이브러리 및 CSV 불러오기
accounts = pd.read_csv(
    r'C:\Machine_Learning_worksheet\Product_Analysis\데이터\1,2,4장\ravenstack_accounts.csv'
)

subscriptions = pd.read_csv(
    r'C:\Machine_Learning_worksheet\Product_Analysis\데이터\1,2,4장\ravenstack_subscriptions.csv'
)

feature_usage = pd.read_csv(
    r'C:\Machine_Learning_worksheet\Product_Analysis\데이터\1,2,4장\ravenstack_feature_usage.csv'
)


# 3. 데이터 크기와 상위 5개 행 확인
print("=== accounts ===")
print(accounts.shape)
display(accounts.head())

print("=== subscriptions ===")
print(subscriptions.shape)
display(subscriptions.head())

print("=== feature_usage ===")
print(feature_usage.shape)
display(feature_usage.head())


# 4. 자료형과 결측치 확인
for name, df in [
    ('accounts', accounts),
    ('subscriptions', subscriptions),
    ('feature_usage', feature_usage)
]:
    print(f"\n=== {name} ===")
    print(pd.DataFrame({
        'dtype': df.dtypes,
        'missing': df.isna().sum()
    }))


# 5. 날짜형 변환
accounts['signup_date'] = pd.to_datetime(
    accounts['signup_date'], errors='coerce'
)

subscriptions['start_date'] = pd.to_datetime(
    subscriptions['start_date'], errors='coerce'
)

subscriptions['end_date'] = pd.to_datetime(
    subscriptions['end_date'], errors='coerce'
)

feature_usage['usage_date'] = pd.to_datetime(
    feature_usage['usage_date'], errors='coerce'
)


# 6. usage_id 중복 개수 확인
duplicate_usage_id = feature_usage['usage_id'].duplicated().sum()

print("usage_id 중복 개수:", duplicate_usage_id)


# 7. usage_count 기초 통계량
display(feature_usage['usage_count'].describe())

=== accounts ===
(500, 10)


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


=== subscriptions ===
(5000, 14)


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


=== feature_usage ===
(25000, 8)


,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,False



=== accounts ===
                 dtype  missing
account_id         str        0
account_name       str        0
industry           str        0
country            str        0
signup_date        str        0
referral_source    str        0
plan_tier          str        0
seats            int64        0
is_trial          bool        0
churn_flag        bool        0

=== subscriptions ===
                   dtype  missing
subscription_id      str        0
account_id           str        0
start_date           str        0
end_date             str     4514
plan_tier            str        0
seats              int64        0
mrr_amount         int64        0
arr_amount         int64        0
is_trial            bool        0
upgrade_flag        bool        0
downgrade_flag      bool        0
churn_flag          bool        0
billing_frequency    str        0
auto_renew_flag     bool        0

=== feature_usage ===
                     dtype  missing
usage_id               str        0
su

count    25000.000000
mean        10.021000
std          3.143729
min          0.000000
25%          8.000000
50%         10.000000
75%         12.000000
max         26.000000
Name: usage_count, dtype: float64

---

## 필수 1. Ravenstack의 AARRR 지표 지도 만들기

### 문제 1-1. 데이터에서 확인 가능한 단계별 대표 지표 계산하기

#### 문제 설명

Ravenstack의 세 테이블을 이용하여 AARRR 단계별 대표 지표를 계산하고, 제공된 데이터만으로 직접 측정하기 어려운 단계도 확인하세요.

이번 문제에서는 다음 기준을 사용합니다.

- Acquisition: 가입한 고유 고객사 수
- Activation: 전체 구독 중 기능 사용 기록이 한 번 이상 있는 구독의 비율
- Retention: 전체 구독 중 현재 `churn_flag`가 `False`인 구독의 비율
- Referral: 제공된 데이터로 직접 계산할 수 없음
- Revenue: 종료일이 없고 체험 구독이 아닌 현재 활성 유료 구독의 MRR 합계

> 여기에서 Retention은 학습을 위한 간단한 대리 지표입니다. 특정 기간 후 다시 사용한 비율을 계산한 정식 리텐션과는 다릅니다.

#### 요구사항

1. 가입한 고유 고객사 수를 `acquired_accounts`로 계산하세요.
2. 기능 사용 기록이 있는 고유 구독 수를 전체 구독 수로 나누어 `activation_rate`를 계산하세요.
3. `churn_flag`가 `False`인 구독의 비율을 `non_churned_rate`로 계산하세요.
4. 종료일이 없고 체험 구독이 아닌 구독의 MRR 합계를 `current_paid_mrr`로 계산하세요.
5. 네 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 각 지표를 AARRR 단계와 연결하고 분석 목적을 설명하세요.
7. Referral 단계를 직접 측정하려면 어떤 이벤트가 추가로 필요한지 제안하세요.

#### 해석 질문

**Q1.** 기능 사용 기록이 있는 구독 비율은 어느 AARRR 단계와 연결되나요?  
**Q2.** `non_churned_rate`를 정식 리텐션율과 동일하게 보면 안 되는 이유는 무엇인가요?  
**Q3.** `referral_source`만으로 Referral 행동을 직접 측정할 수 있나요?  
**Q4.** MRR은 어느 AARRR 단계와 연결되나요?

#### 제출 결과

- AARRR 단계별 대표 지표 계산 코드와 결과
- 각 지표의 단계 및 분석 목적
- Referral 측정에 필요한 이벤트 제안
- Q1~Q4 답변

In [3]:
# 필수 1 코드를 작성하세요.

# 1. Acquisition
acquired_accounts = accounts['account_id'].nunique()

# 2. Activation
activation_rate = (
    feature_usage['subscription_id'].nunique()
    / subscriptions['subscription_id'].nunique()
)

# 3. Retention
non_churned_rate = (subscriptions['churn_flag'] == 0).mean()

# 4. Revenue
current_paid_mrr = subscriptions.loc[
    subscriptions['end_date'].isna() &
    (subscriptions['mrr_amount'] > 0),
    'mrr_amount'
].sum()

# 5. 결과 출력
print(f"Acquisition : {acquired_accounts:,}")
print(f"Activation  : {activation_rate:.2%}")
print(f"Retention   : {non_churned_rate:.2%}")
print(f"Revenue     : {current_paid_mrr:,.0f}")

Acquisition : 500
Activation  : 99.34%
Retention   : 90.28%
Revenue     : 10,159,608


### 필수 1 답변 작성란

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**

---

## 필수 2. 요금제별 Activation 지표 비교하기

### 문제 2-1. 2024년 12월 제품 활성률을 요금제별로 비교하기

#### 문제 설명

프로덕트팀은 다음 질문에 답하려고 합니다.

> 2024년 12월에 어떤 요금제의 제품 활성률이 상대적으로 낮은가?

이 질문은 사용자를 요금제별로 나누어 비교하므로 **세그멘테이션 분석**에 해당하며, AARRR 중 Activation 단계를 살펴봅니다.

이번 실습에서 제품 활성률은 다음과 같이 정의합니다.

> 12월 제품 활성 유료 구독 수 ÷ 12월 이용 가능 유료 구독 수

#### 요구사항

1. 2024년 12월의 시작일과 마지막 날을 설정하세요.
2. 12월에 이용 가능한 유료 구독을 `eligible_dec`에 저장하세요.
3. 12월에 기능을 사용한 구독 ID를 `active_subscription_ids`로 준비하세요.
4. `eligible_dec`에 `is_product_active` 컬럼을 만드세요.
5. `plan_tier`별로 다음 값을 집계하여 `plan_activation`을 만드세요.
   - 이용 가능 유료 구독 수
   - 제품 활성 유료 구독 수
   - 제품 활성률
6. 제품 활성률이 낮은 순서로 정렬하고 백분율로 출력하세요.
7. 가장 낮은 요금제를 찾아 개선을 위해 추가로 확인할 내용을 제안하세요.
8. 이 질문에 세그멘테이션 분석이 적절한 이유를 설명하세요.

#### 해석 질문

**Q1.** 이 분석은 AARRR 중 어느 단계에 해당하나요?  
**Q2.** 어떤 분석 기법을 사용했으며, 그 이유는 무엇인가요?  
**Q3.** 제품 활성률이 가장 낮은 요금제는 무엇인가요?  
**Q4.** 해당 요금제에서 추가로 어떤 데이터를 확인하면 좋을까요?

#### 제출 결과

- `plan_activation` 집계 코드와 결과
- AARRR 단계와 분석 기법
- 우선 점검할 요금제
- 추가 확인 데이터
- Q1~Q4 답변

In [4]:
# 필수 2 코드를 작성하세요.
# 1. 2024년 12월 시작/마지막 날짜
dec_start = pd.Timestamp('2024-12-01')
dec_end = pd.Timestamp('2024-12-31')


# 2. 12월 이용 가능한 유료 구독
eligible_dec = subscriptions[
    (subscriptions['start_date'] <= dec_end) &
    (subscriptions['end_date'].isna() | (subscriptions['end_date'] >= dec_start)) &
    (subscriptions['mrr_amount'] > 0)
].copy()


# 3. 12월 기능 사용 구독 ID
active_subscription_ids = feature_usage.loc[
    feature_usage['usage_date'].between(dec_start, dec_end),
    'subscription_id'
].unique()


# 4. 제품 활성 여부
eligible_dec['is_product_active'] = (
    eligible_dec['subscription_id'].isin(active_subscription_ids)
)


# 5. 요금제별 활성화율
plan_activation = eligible_dec.groupby('plan_tier').agg(
    eligible_subscriptions=('subscription_id', 'nunique'),
    active_subscriptions=('is_product_active', 'sum')
)

plan_activation['activation_rate'] = (
    plan_activation['active_subscriptions']
    / plan_activation['eligible_subscriptions']
)


# 6. 낮은 순서로 출력
plan_activation = plan_activation.sort_values('activation_rate')

plan_activation['activation_rate'] = (
    plan_activation['activation_rate'] * 100
).round(2)

display(plan_activation)

,eligible_subscriptions,active_subscriptions,activation_rate
plan_tier,,,
Pro,1342,260,19.37
Basic,1267,247,19.49
Enterprise,1372,281,20.48


### 필수 2 답변 작성란

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**

---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 결제 주기별 Activation 지표 비교하기

#### 문제 설명

프로덕트팀은 2024년 12월 제품 활성률이 월간 결제 구독과 연간 결제 구독에서 다르게 나타나는지 확인하려고 합니다.

> 이 과제는 필수 2의 요금제별 세그멘테이션을 `billing_frequency`에 동일하게 적용하는 문제입니다.

#### 요구사항

1. 필수 2에서 만든 `eligible_dec`과 `is_product_active`를 사용하세요.
2. `billing_frequency`별로 다음 값을 집계하여 `billing_activation`을 만드세요.
   - 이용 가능 유료 구독 수
   - 제품 활성 유료 구독 수
   - 제품 활성률
3. 제품 활성률이 낮은 순서로 정렬하고 백분율로 출력하세요.
4. 제품 활성률이 더 낮은 결제 주기를 찾으세요.
5. 해당 그룹을 우선 점검 대상으로 정하고 추가로 확인할 내용을 한 가지 제안하세요.
6. 이 분석의 AARRR 단계와 분석 기법을 설명하세요.

#### 해석 질문

**Q1.** 제품 활성률이 더 낮은 결제 주기는 무엇인가요?  
**Q2.** 이 분석은 어떤 AARRR 단계와 분석 기법에 해당하나요?  
**Q3.** 두 그룹의 제품 활성률 차이만으로 결제 주기가 낮은 활성의 원인이라고 결론 내릴 수 있나요?

**Q4.** 퍼널 분석·코호트 리텐션 분석·세그멘테이션을 표로 정리하세요. 각 분석의 계산 또는 분류 원리, 해결하려는 분석 질문, 관련 AARRR 단계를 설명하세요. 전환율·이탈률의 분모, 코호트의 시작 기준과 경과 기간, 세그먼트 분류 기준을 포함하세요.

#### 제출 결과

- `billing_activation` 집계 코드와 결과
- 우선 점검할 결제 주기
- 추가 확인 내용
- AARRR 단계와 분석 기법
- Q1~Q4 답변

In [5]:
# 과제 1 코드를 작성하세요.

# ============================================================
# 문제 3-1. 결제 주기별 Activation 지표 비교
# ============================================================

# billing_frequency별 집계
billing_activation = (
    eligible_dec
    .groupby('billing_frequency')
    .agg(
        eligible_subscriptions=('subscription_id', 'nunique'),
        active_subscriptions=('is_product_active', 'sum')
    )
    .reset_index()
)

# 제품 활성률 계산
billing_activation['activation_rate'] = (
    billing_activation['active_subscriptions']
    / billing_activation['eligible_subscriptions']
    * 100
)

# 활성률이 낮은 순서로 정렬
billing_activation = (
    billing_activation
    .sort_values('activation_rate')
    .reset_index(drop=True)
)

# 소수점 둘째 자리까지
billing_activation['activation_rate'] = (
    billing_activation['activation_rate'].round(2)
)

display(billing_activation)


# 활성률이 가장 낮은 결제 주기
lowest_billing = billing_activation.iloc[0]

print("우선 점검 대상:", lowest_billing['billing_frequency'])
print(f"제품 활성률: {lowest_billing['activation_rate']:.2f}%")

,billing_frequency,eligible_subscriptions,active_subscriptions,activation_rate
0,annual,1968,385,19.56
1,monthly,2013,403,20.02


우선 점검 대상: annual
제품 활성률: 19.56%


### 과제 1 답변 작성란

**Q1.** 제품 활성률이 더 낮은 결제 주기는 무엇인가요?  

annual

**Q2.** 이 분석은 어떤 AARRR 단계와 분석 기법에 해당하나요?  

Activation, 결제 주기별로 고객을 나누어 비교하는 세그멘테이션 분석기법

**Q3.** 두 그룹의 제품 활성률 차이만으로 결제 주기가 낮은 활성의 원인이라고 결론 내릴 수 있나요?

아니요 현재 이 서비스가 어떻게 제공을 하는건지는 모르기 때문에 

**Q4.** 퍼널 분석·코호트 리텐션 분석·세그멘테이션을 표로 정리하세요. 각 분석의 계산 또는 분류 원리, 해결하려는 분석 질문, 관련 AARRR 단계를 설명하세요. 전환율·이탈률의 분모, 코호트의 시작 기준과 경과 기간, 세그먼트 분류 기준을 포함하세요.

| 분석      | 원리                                      | 분석 질문               | AARRR                      |
| ------- | --------------------------------------- | ------------------- | -------------------------- |
| 퍼널 분석   | 단계별 이전 단계 인원을 분모로 전환율·이탈률 계산            | 어느 단계에서 많이 이탈하는가?   | Acquisition → Activation 등 |
| 코호트 리텐션 | 같은 시작 시점의 고객을 묶고 M1, M2 등 경과 기간별 잔존율 비교 | 시간이 지나도 계속 사용하는가?   | Retention                  |
| 세그멘테이션  | 요금제, 결제 주기 등 기준으로 고객을 나누어 지표 비교         | 어떤 고객 그룹에서 차이가 나는가? | 여러 단계에 적용 가능               |


---

## 실습 마무리

아래 질문에 답하세요.

1. Ravenstack의 사용자 여정을 어떤 AARRR 단계로 나누었나요?
2. 각 단계는 어떤 분석 질문에 답하기 위해 사용했나요?
3. 제공된 데이터에서 직접 측정하기 어려운 단계는 무엇이었나요?
4. 요금제와 결제 주기별 비교에는 어떤 분석 기법을 사용했나요?
5. 분석 결과를 근거로 어떤 대상을 우선 점검했나요?